In [1]:
#../a/b.py --> for saving

In [2]:
import os
notebook_dir = os.getcwd() 

In [3]:
import sys
loc = os.path.abspath(os.path.join(notebook_dir, '..', '..', 'src'))
sys.path.append(loc)

In [4]:
# other basic imports
import time
import torch

In [5]:
from gnm import defaults, utils, evaluation, fitting, generative_rules, weight_criteria

In [6]:
from gnm import *

In [7]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [8]:
distance_matrix = defaults.get_distance_matrix(device=DEVICE) #create my own distance matrix?

In [ ]:
import numpy as np

file_path = "/Users/anchalbhaskar/Documents/Projects/GenerativeNetworkModels/weighted_connectivity.npy"
weighted_connectivity = np.load(file_path)

print(weighted_connectivity.shape)

(90, 90, 446)


In [ ]:
sns.heatmap(weighted_connectivity[:, :, 0])

NameError: name 'sns' is not defined

In [ ]:
print("Original shape:", weighted_connectivity.shape)

# (90, 90, 446) → (446, 90, 90)
all_networks_np = np.transpose(weighted_connectivity, (2, 0, 1))
print("After transpose:", all_networks_np.shape)   # (446, 90, 90)

# 0 stays 0, anything > 0 becomes 1
all_binary_networks_np = (all_networks_np > 0).astype(np.float32)

print("Binary shape:", all_binary_networks_np.shape)  # (446, 90, 90)

all_binary_networks = torch.tensor(all_binary_networks_np, device=DEVICE)

S, N, _ = all_binary_networks.shape
print("Num subjects:", S)  #  446
print("Num nodes:", N)     # 90


NameError: name 'weighted_connectivity' is not defined

In [ ]:
import seaborn as sns

sns.heatmap(all_binary_networks[0, :, :])

NameError: name 'all_binary_networks' is not defined

In [ ]:
num_connections = int(all_binary_networks.sum().item() / 446 )
print(f"The binary consensus network contains {num_connections} connections.")

The binary consensus network contains 800 connections.


In [ ]:
num_simulations = 1

In [ ]:
file_path = "/Users/anchalbhaskar/Documents/Projects/GenerativeNetworkModels/seed.npy"
seed = np.load(file_path)
seed_new = np.broadcast_to(seed, (num_simulations,seed.shape[0], seed.shape[1]))
tensor_seed = torch.tensor(seed_new, device=DEVICE)
#sns.heatmap(tensor_seed)

In [ ]:
#Sweep --> make a consensus network on a single brain then decide on the eta and gamma values + create a window 
eta_values   = torch.linspace(-5.0, 0.0, 25)   
gamma_values = torch.linspace( 0.0, 1.0, 25)   

binary_sweep_parameters = fitting.BinarySweepParameters(
    eta = eta_values,
    gamma = gamma_values,
    lambdah = torch.Tensor([0.0]),
    distance_relationship_type = ["powerlaw"],
    preferential_relationship_type = ["powerlaw"],
    heterochronicity_relationship_type = ["powerlaw"],
    generative_rule = [generative_rules.MatchingIndex()],
    num_iterations = [num_connections],
)

#weighted_sweep_parameters = fitting.WeightedSweepParameters(
    #alpha = [0.01],
    #optimisation_criterion = [
        #weight_criteria.DistanceWeightedCommunicability(
            #distance_matrix=distance_matrix
        #)
    #],
#)

sweep_config = fitting.SweepConfig(
    binary_sweep_parameters = binary_sweep_parameters,
    #weighted_sweep_parameters = weighted_sweep_parameters,
    num_simulations = num_simulations,
    distance_matrix = [distance_matrix],
    seed_adjacency_matrix = tensor_seed
)

In [ ]:
criteria = [ evaluation.ClusteringKS(), evaluation.DegreeKS(), evaluation.EdgeLengthKS(distance_matrix) ]
energy = evaluation.MaxCriteria( criteria )
binary_evaluations = [energy]

In [ ]:
start_time = time.perf_counter()

In [ ]:
experiments = fitting.perform_sweep(sweep_config=sweep_config, 
                                binary_evaluations=binary_evaluations, 
                                real_binary_matrices= all_binary_networks,
                                #weighted_evaluations=weighted_evaluations,
                                save_model = False,
                                save_run_history = False,
                                verbose=True,
                                device=torch.device('cpu')
)

end_time = time.perf_counter()

Using device: cpu for GNM simulations


Configuration Iterations: 100%|██████████| 625/625 [1:10:26<00:00,  6.76s/it]  

Average time per run: 1.69 seconds
Total time for sweep: 1055.30 seconds


In [ ]:
print(f"Sweep took {end_time - start_time:0.3f} seconds.")

total_simulations = num_simulations * len(eta_values) * len(gamma_values)

print(f"Total number of simulations: {total_simulations}")

print(f"Average time per simulation: {(end_time - start_time) / total_simulations:0.3f} seconds.")

Sweep took 213.433 seconds.
Total number of simulations: 360
Average time per simulation: 0.593 seconds.


In [ ]:
optimal_experiments, optimal_energies = fitting.optimise_evaluation(
    experiments=experiments,
    criterion=energy,
)

optimal_experiment = optimal_experiments[0]
optimal_energy = optimal_energies[0]

In [ ]:
print(f"Optimal energy: {optimal_energy:0.3f}")
print(f"Optimal value of eta: {optimal_experiment.run_config.binary_parameters.eta:0.2f}")
print(f"Optimal value of gamma: {optimal_experiment.run_config.binary_parameters.gamma:0.2f}")

Optimal energy: 0.486
Optimal value of eta: -2.00
Optimal value of gamma: 0.40


In [ ]:
#print(all_binary_networks.shape)

P, N, N = all_binary_networks.shape

for subj_idx in range(P):

    print(f"\nRunning sweep for subject {subj_idx}...")

    # Get this subject's adjacency matrix
    A_real = all_binary_networks[subj_idx]   # (90, 90)

    experiments = fitting.perform_sweep(
        sweep_config = sweep_config,
        binary_evaluations = binary_evaluations,
        real_binary_matrices = A_real,
        save_model = False,
        save_run_history = False,
    )


Running sweep for subject 0...


TypeCheckError: Type-check error whilst checking the parameters of gnm.fitting.sweep.perform_sweep.
The problem arose whilst typechecking parameter 'real_binary_matrices'.
Actual value: tensor([[0., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 1., 0., 0.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 1., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 1., 0., 0.]])
Expected type: typing.Optional[Float[Tensor, 'num_real_binary_networks num_nodes num_nodes']].
----------------------
Called with parameters: { 'binary_evaluations': [ <gnm.evaluation.composite_criteria.MaxCriteria object at 0x3227c7280>],
  'device': None,
  'method': 'grid',
  'metric_to_optimise': None,
  'num_bayesian_runs': 30,
  'real_binary_matrices': tensor([[0., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 1., 0., 0.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 1., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ...,